In [1]:
import pandas as pd

In [2]:
!pip install neo4j


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
data_dict = pd.read_csv('data_dict_final.csv')

In [5]:
nacc_data = pd.read_csv('nacc_final_data.csv')

In [17]:
nacc_data.head()

,NACCID,SEX,EDUC,NACCLIVS,INDEPEND,NACCFAM,NACCMOM,NACCDAD,ANYMEDS,TOBAC30,...,NACCEMD,NACCEPMD,NACCHTNC,NACCLIPL,NACCNSD,NACCPDMD,NACCVASD,NACCBMI,NACCUDSD,VISIT_DATE
0,NACC002909,1,16.0,4.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,32.4,3,2022-12-28
1,NACC002909,1,16.0,2.0,1.0,1.0,0.0,0.0,1.0,NaN,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,30.7,3,2024-01-23
2,NACC003487,1,16.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.7,1,2023-11-15
3,NACC004352,2,16.0,2.0,2.0,NaN,NaN,NaN,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,4,2021-10-05
4,NACC004687,1,12.0,1.0,1.0,NaN,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.0,1,2022-11-14


In [47]:
nacc_data.isna().sum()

NACCID            0
SEX               0
EDUC            974
NACCLIVS        324
INDEPEND        597
              ...  
NACCPDMD       2739
NACCVASD       2739
NACCBMI       39645
NACCUDSD          0
VISIT_DATE        0
Length: 177, dtype: int64

In [6]:
nacc_data.shape

(195196, 177)

In [7]:
from neo4j import GraphDatabase

In [8]:
URI = "bolt://13.218.45.200"
USERNAME = "neo4j"
PASSWORD = "comment-regulation-races"

In [9]:
# Connect to Neo4j
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

In [10]:
data_dict.head()

,VariableName,Form,VariableType,ShortDescriptor,DataType,AllowableCodes
0,NACCID,header,NACC derived variable,Subject ID number,Numeric longitudinal,Prefix NACC followed by 0 - 10 numbers
1,SEX,a1,Original UDS question,Subject's sex,Numeric cross-sectional,1 = Male\n2 = Female
2,EDUC,a1,Original UDS question,Years of education,Numeric cross-sectional,0 - 36\n99 = Unknown
3,NACCLIVS,a1,NACC derived variable,Living situation,Numeric longitudinal,1 = Lives alone\n2 = Lives with spouse or par...
4,INDEPEND,a1,Original UDS question,Level of independence,Numeric longitudinal,1 = Able to live independently \n2 = Requires...


In [12]:
data_dict['Form'].unique()

array(['header', 'a1', 'a3', 'a4', 'a5', 'b1', 'b4', 'b5', 'b6', 'b7',
       'b9', 'c1c2c2t', 'c1', 'd1'], dtype=object)

In [38]:
form_map = {
    'header': 'header',
    'a1':'Demographics',
    'a3':'Family History',
    'a4':'Medications',
    'a5':'Health History',
    'b1':'Physical',
    'b4':'CDR Plus NACC FTLD',
    'b5':'Neuropsychiatric Inventory Questionnaire',
    'b6':'Geriatric Depression Scale',
    'b7':'Functional Activities Questionnaire',
    'b9':'Clinician Judgment of Symptoms',
    'c1c2c2t':'Neuropsychological battery Summary Scores',
    'c1':'Neuropsychological battery Summary Scores',
    'c2':'Neuropsychological battery Summary Scores',
    'c2t':'Neuropsychological battery Summary Scores',
    'd1':'Clinician Diagnosis'
}

In [39]:
data_dict['Form_name'] = data_dict['Form'].map(form_map)

In [40]:
data_dict.head()

,VariableName,Form,VariableType,ShortDescriptor,DataType,AllowableCodes,Form_name
0,NACCID,header,NACC derived variable,Subject ID number,Numeric longitudinal,Prefix NACC followed by 0 - 10 numbers,header
1,SEX,a1,Original UDS question,Subject's sex,Numeric cross-sectional,1 = Male\n2 = Female,Demographics
2,EDUC,a1,Original UDS question,Years of education,Numeric cross-sectional,0 - 36\n99 = Unknown,Demographics
3,NACCLIVS,a1,NACC derived variable,Living situation,Numeric longitudinal,1 = Lives alone\n2 = Lives with spouse or par...,Demographics
4,INDEPEND,a1,Original UDS question,Level of independence,Numeric longitudinal,1 = Able to live independently \n2 = Requires...,Demographics


In [41]:
form_dict = {}
for i in data_dict['Form_name'].unique():
    if i != 'header':
        form_dict[i] = data_dict[data_dict['Form_name']==i]['VariableName'].to_list() 
    else:
        form_dict[i] = data_dict[data_dict['Form_name']==i]['VariableName'].to_list() + ['VISIT_DATE']

In [71]:
data_dict.loc[data_dict['VariableName'].isin(form_dict['Clinician Judgment of Symptoms']), 'ShortDescriptor'].iloc[3]

'Indicate whether the subject currently is meaningfully impaired, relative to previously attained abilities, in executive function - judgment, planning, or problem-solving'

In [72]:
data_dict[data_dict['VariableName'].isin(form_dict['Neuropsychological battery Summary Scores'])]

,VariableName,Form,VariableType,ShortDescriptor,DataType,AllowableCodes,Form_name
121,COGSTAT,c1c2c2t,Original UDS question,"Per the clinician, based on the UDS \nneuropsy...",Numeric longitudinal,0 = Clinician unable to render opinion\t\n1 =...,Neuropsychological battery Summary Scores
122,NACCMMSE,c1,NACC derived variable,Total MMSE score (using D-L-R-O-W),Numeric longitudinal,0 - 30\n88 = Score not calculated; missing at...,Neuropsychological battery Summary Scores
123,BOSTON,c1,Original UDS question,Boston Naming Test (30) - Total score,Numeric longitudinal,0 - 30\n95 = Physical problem\n96 = Cognitive...,Neuropsychological battery Summary Scores


In [43]:
form_dict.keys()

dict_keys(['header', 'Demographics', 'Family History', 'Medications', 'Health History', 'Physical', 'CDR Plus NACC FTLD', 'Neuropsychiatric Inventory Questionnaire', 'Geriatric Depression Scale', 'Functional Activities Questionnaire', 'Clinician Judgment of Symptoms', 'Neuropsychological battery Summary Scores', 'Clinician Diagnosis'])

In [51]:
class NACCGraphLoader:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def load_section_to_graph(self, df, section_name, column_list):
        rows = []

        for _, row in df.iterrows():
            values = []
            for col in column_list:
                val = row.get(col)
                if pd.notnull(val):
                    values.append({"name": col, 
                                   "value": val,
                                   "description": description_lookup.get(col, "")
                                   
                                  })
            if values:
                rows.append({
                    "patient_id": row["NACCID"],
                    "visit_date": row["VISIT_DATE"],
                    "items": values
                })

        if not rows:
            print(f"No valid rows for section: {section_name}")
            return

        label = section_name.replace(" ", "")  # e.g., 'Clinician Diagnosis' -> 'ClinicianDiagnosis'
        rel = f"HAS_{label.upper()}"

        query = f"""
        UNWIND $rows AS row
        MERGE (p:Patient {{id: row.patient_id}})
        MERGE (v:Visit {{date: row.visit_date, patient_id: row.patient_id}})
        MERGE (p)-[:HAS_VISIT]->(v)
        WITH row, v
        UNWIND row.items AS item
        WITH v, item WHERE item.value IS NOT NULL
        MERGE (n:{label} {{name: item.name}})
        SET n.value = item.value
        MERGE (v)-[:{rel}]->(n)
        """

        with self.driver.session(database="neo4j") as session:
            session.run(query, rows=rows)

In [52]:
loader = NACCGraphLoader(URI, USERNAME, PASSWORD)
try:
    for section in list(form_dict.keys())[1:]:
        if form_dict[section]:
            print(f"Loading section: {section}")
            loader.load_section_to_graph(nacc_data.head(100), section, form_dict[section])
    print("All sections loaded successfully!")

finally:
    loader.close()

Loading section: Demographics
Loading section: Family History
Loading section: Medications
Loading section: Health History
Loading section: Physical
Loading section: CDR Plus NACC FTLD
Loading section: Neuropsychiatric Inventory Questionnaire
Loading section: Geriatric Depression Scale
Loading section: Functional Activities Questionnaire
Loading section: Clinician Judgment of Symptoms
Loading section: Neuropsychological battery Summary Scores
Loading section: Clinician Diagnosis
All sections loaded successfully!
